# 🎨 Free Image + Video Generator (Chromebook friendly)

**Before pressing play:** click **Runtime → Change runtime type → T4 GPU → Save**.

Then press the ▶️ button below and wait. Free Colab gives a few GPU hours per day; when it stops, just press play again tomorrow.

In [ ]:
#@title ▶️ Press the play button on the left, then wait ~10 minutes. A link will appear at the bottom.
import os, subprocess, re, time, threading, sys

def sh(cmd):
    subprocess.run(cmd, shell=True, check=True)

print("1/4  Installing ComfyUI ...")
if not os.path.exists("/content/ComfyUI"):
    sh("cd /content && git clone -q https://github.com/comfyanonymous/ComfyUI.git")
sh("cd /content/ComfyUI && pip install -q -r requirements.txt")
sh("cd /content/ComfyUI/custom_nodes && (test -d ComfyUI-Manager || git clone -q https://github.com/ltdrdata/ComfyUI-Manager.git)")

print("2/4  Downloading the image model (SDXL, ~7 GB) ...")
M = "/content/ComfyUI/models"
def dl(url, dest):
    if not os.path.exists(dest):
        sh(f'wget -q --show-progress -O "{dest}" "{url}"')
dl("https://huggingface.co/stabilityai/stable-diffusion-xl-base-1.0/resolve/main/sd_xl_base_1.0.safetensors",
   f"{M}/checkpoints/sd_xl_base_1.0.safetensors")

print("3/4  Downloading the video model (LTX-Video, ~11 GB) ...")
dl("https://huggingface.co/Lightricks/LTX-Video/resolve/main/ltx-video-2b-v0.9.5.safetensors",
   f"{M}/checkpoints/ltx-video-2b-v0.9.5.safetensors")
os.makedirs(f"{M}/text_encoders", exist_ok=True)
dl("https://huggingface.co/comfyanonymous/flux_text_encoders/resolve/main/t5xxl_fp8_e4m3fn.safetensors",
   f"{M}/text_encoders/t5xxl_fp8_e4m3fn.safetensors")

print("4/4  Starting the app and creating your link ...")

def ready():
    import urllib.request
    while True:
        try:
            urllib.request.urlopen("http://127.0.0.1:8188/", timeout=2); break
        except Exception:
            time.sleep(2)
    from google.colab.output import eval_js
    url = eval_js("google.colab.kernel.proxyPort(8188)")
    print("\n" + "="*60)
    print("✅ YOUR GENERATOR IS READY. Click this link:")
    print("   " + url)
    print("="*60)
    print("Inside the app: click  Workflow > Browse Templates")
    print("  • Images: pick 'SDXL' , type a prompt, press Run")
    print("  • Video : pick 'LTX-Video' (text to video), press Run")
    print("Keep this Colab tab open while you use it.\n")

threading.Thread(target=ready, daemon=True).start()
sh("cd /content/ComfyUI && python main.py --listen 127.0.0.1 --port 8188 --lowvram --preview-method auto --enable-cors-header '*'")
